# Idea C: Quantization Impact Experiments (Colab T4)

Runs 25 model×quant variants × 500 pages on a T4 GPU.

**Setup:** Runtime → Change runtime type → **T4 GPU** → Run All.

**Resume:** Results persist on Google Drive. Re-run picks up where the previous session stopped.

**Disk strategy:** Downloads one GGUF at a time, deletes after processing. Max ~12GB disk usage.

**Estimated total:** 15-22h across 2-3 sessions (12h each).

In [ ]:
# Cell 1: Platform detection, Google Drive mount, env vars
import os
import sys
from pathlib import Path

# Detect platform
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle")
PLATFORM = "colab" if IN_COLAB else ("kaggle" if IN_KAGGLE else "local")
print(f"Platform: {PLATFORM}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/iitsrc")
elif IN_KAGGLE:
    DRIVE_BASE = Path("/kaggle/working/iitsrc")
else:
    DRIVE_BASE = Path.cwd() / "iitsrc_output"

# Persistent dirs on Drive
DATA_DIR = DRIVE_BASE / "data" / "scrapegraphai"
RESULTS_DIR = DRIVE_BASE / "results" / "idea-c"
MODELS_DIR = Path("/content/models") if IN_COLAB else Path("/kaggle/working/models") if IN_KAGGLE else Path("./models")

for d in [DATA_DIR, RESULTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Set env vars for shared modules
os.environ["IITSRC_DATA_DIR"] = str(DATA_DIR)
os.environ["IITSRC_MODELS_DIR"] = str(MODELS_DIR)
os.environ["IITSRC_RESULTS_DIR"] = str(RESULTS_DIR)

print(f"Data:    {DATA_DIR}")
print(f"Models:  {MODELS_DIR}")
print(f"Results: {RESULTS_DIR}")

# GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU detected!"

In [ ]:
# Cell 2: Install dependencies
# Pre-built CUDA 12.4 wheel (~30s) — much faster than source build (~15min)
!pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
    --quiet 2>&1 | tail -3

# If the above fails, uncomment for source build:
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir

!pip install rapidfuzz tiktoken tqdm pandas numpy datasets huggingface-hub jsonschema --quiet 2>&1 | tail -3

# Verify llama.cpp sees CUDA
import llama_cpp
print(f"llama-cpp-python {llama_cpp.__version__}")

In [ ]:
# Cell 3: Clone repo, add to sys.path
import subprocess

REPO_DIR = Path("/content/iitsrc") if IN_COLAB else Path("/kaggle/working/iitsrc") if IN_KAGGLE else Path("./iitsrc")

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/andrejvysny/iitsrc.git",
        str(REPO_DIR),
    ], check=True)
    print(f"Cloned to {REPO_DIR}")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    print(f"Repo exists at {REPO_DIR}")

# Add to path
for p in [str(REPO_DIR), str(REPO_DIR / "idea-c-quantization" / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify imports
from shared.dataset import load_and_cache_sample, DATA_DIR as LOADED_DATA_DIR
from shared.metrics import compute_all_metrics
from inference import (
    load_and_extract, get_all_variants, get_model_path,
    MODEL_QUANTS, MODEL_FAMILIES, MODELS_DIR as LOADED_MODELS_DIR,
    unload_model,
)
from shared.utils import count_tokens

print(f"Imports OK — DATA_DIR={LOADED_DATA_DIR}, MODELS_DIR={LOADED_MODELS_DIR}")

In [ ]:
# Cell 4: Load 500-page dataset (cached on Drive)
records = load_and_cache_sample(500)
print(f"Loaded {len(records)} records")
print(f"Sample keys: {list(records[0].keys())}")
print(f"Sample schema props: {len(records[0]['schema'].get('properties', {}))}")

In [ ]:
# Cell 5: T4 benchmark — download smallest model, run 5 pages
import time
from huggingface_hub import hf_hub_download

# Smallest variant: qwen2.5-3b q2_k (~1.2 GB)
BENCH_MODEL = "qwen2.5-3b"
BENCH_QUANT = "q2_k"
BENCH_PAGES = 5

bench_path = get_model_path(BENCH_MODEL, BENCH_QUANT)
if not bench_path.exists():
    print(f"Downloading {BENCH_MODEL}/{BENCH_QUANT} for benchmark...")
    bench_path.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(
        repo_id="bartowski/Qwen2.5-3B-Instruct-GGUF",
        filename="Qwen2.5-3B-Instruct-Q2_K.gguf",
        local_dir=str(bench_path.parent),
        local_dir_use_symlinks=False,
    )
    # Rename to match expected filename
    expected_name = bench_path.name
    downloaded = bench_path.parent / "Qwen2.5-3B-Instruct-Q2_K.gguf"
    if downloaded.exists() and not bench_path.exists():
        downloaded.rename(bench_path)
    print(f"Downloaded: {bench_path.stat().st_size / 1e9:.1f} GB")

# Benchmark
from run_experiments import truncate_to_tokens

latencies = []
for i, page in enumerate(records[:BENCH_PAGES]):
    schema = page["schema"]
    content = truncate_to_tokens(page["content_markdown"], 3500)
    t0 = time.perf_counter()
    result = load_and_extract(BENCH_MODEL, BENCH_QUANT, content, schema)
    elapsed = time.perf_counter() - t0
    latencies.append(elapsed)
    toks = result.get("tokens_out", 0)
    print(f"  Page {i+1}: {elapsed:.1f}s, {toks} tokens out")

unload_model()

avg = sum(latencies) / len(latencies)
total_variants = sum(len(q) for q in MODEL_QUANTS.values())
total_inferences = total_variants * len(records)
est_hours = (avg * total_inferences) / 3600

print(f"\n--- T4 Benchmark ---")
print(f"Avg latency: {avg:.1f}s/page")
print(f"Total inferences: {total_variants} variants × {len(records)} pages = {total_inferences}")
print(f"Estimated total: {est_hours:.1f}h (optimistic, larger models will be slower)")
print(f"Estimated sessions (12h each): {est_hours / 12:.1f}")

In [ ]:
# Cell 6: Download helpers & disk manager
import shutil
from huggingface_hub import hf_hub_download

# Mirror of download_models.py — (repo_id, hf_filename, local_dir, local_filename)
DOWNLOAD_MAP: dict[tuple[str, str], tuple[str, str, str, str]] = {
    # Qwen2.5-3B
    ("qwen2.5-3b", "f16"):    ("bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen2.5-3B-Instruct-f16.gguf",       "qwen2.5-3b", "qwen2.5-3b-instruct-f16.gguf"),
    ("qwen2.5-3b", "q8_0"):   ("bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen2.5-3B-Instruct-Q8_0.gguf",      "qwen2.5-3b", "qwen2.5-3b-instruct-q8_0.gguf"),
    ("qwen2.5-3b", "q6_k"):   ("bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen2.5-3B-Instruct-Q6_K.gguf",      "qwen2.5-3b", "qwen2.5-3b-instruct-q6_k.gguf"),
    ("qwen2.5-3b", "q5_k_m"): ("bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen2.5-3B-Instruct-Q5_K_M.gguf",    "qwen2.5-3b", "qwen2.5-3b-instruct-q5_k_m.gguf"),
    ("qwen2.5-3b", "q4_k_m"): ("bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen2.5-3B-Instruct-Q4_K_M.gguf",    "qwen2.5-3b", "qwen2.5-3b-instruct-q4_k_m.gguf"),
    ("qwen2.5-3b", "q3_k_m"): ("bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen2.5-3B-Instruct-Q3_K_M.gguf",    "qwen2.5-3b", "qwen2.5-3b-instruct-q3_k_m.gguf"),
    ("qwen2.5-3b", "q2_k"):   ("bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen2.5-3B-Instruct-Q2_K.gguf",      "qwen2.5-3b", "qwen2.5-3b-instruct-q2_k.gguf"),
    # Llama-3.2-3B
    ("llama-3.2-3b", "f16"):    ("bartowski/Llama-3.2-3B-Instruct-GGUF", "Llama-3.2-3B-Instruct-f16.gguf",    "llama-3.2-3b", "llama-3.2-3b-instruct-f16.gguf"),
    ("llama-3.2-3b", "q8_0"):   ("bartowski/Llama-3.2-3B-Instruct-GGUF", "Llama-3.2-3B-Instruct-Q8_0.gguf",   "llama-3.2-3b", "llama-3.2-3b-instruct-q8_0.gguf"),
    ("llama-3.2-3b", "q6_k"):   ("bartowski/Llama-3.2-3B-Instruct-GGUF", "Llama-3.2-3B-Instruct-Q6_K.gguf",   "llama-3.2-3b", "llama-3.2-3b-instruct-q6_k.gguf"),
    ("llama-3.2-3b", "q5_k_m"): ("bartowski/Llama-3.2-3B-Instruct-GGUF", "Llama-3.2-3B-Instruct-Q5_K_M.gguf", "llama-3.2-3b", "llama-3.2-3b-instruct-q5_k_m.gguf"),
    ("llama-3.2-3b", "q4_k_m"): ("bartowski/Llama-3.2-3B-Instruct-GGUF", "Llama-3.2-3B-Instruct-Q4_K_M.gguf", "llama-3.2-3b", "llama-3.2-3b-instruct-q4_k_m.gguf"),
    ("llama-3.2-3b", "q3_k_l"): ("bartowski/Llama-3.2-3B-Instruct-GGUF", "Llama-3.2-3B-Instruct-Q3_K_L.gguf", "llama-3.2-3b", "llama-3.2-3b-instruct-q3_k_l.gguf"),
    # Phi-3.5-mini
    ("phi-3.5-mini", "q8_0"):   ("bartowski/Phi-3.5-mini-instruct-GGUF", "Phi-3.5-mini-instruct-Q8_0.gguf",   "phi-3.5-mini", "phi-3.5-mini-instruct-q8_0.gguf"),
    ("phi-3.5-mini", "q6_k"):   ("bartowski/Phi-3.5-mini-instruct-GGUF", "Phi-3.5-mini-instruct-Q6_K.gguf",   "phi-3.5-mini", "phi-3.5-mini-instruct-q6_k.gguf"),
    ("phi-3.5-mini", "q5_k_m"): ("bartowski/Phi-3.5-mini-instruct-GGUF", "Phi-3.5-mini-instruct-Q5_K_M.gguf", "phi-3.5-mini", "phi-3.5-mini-instruct-q5_k_m.gguf"),
    ("phi-3.5-mini", "q4_k_m"): ("bartowski/Phi-3.5-mini-instruct-GGUF", "Phi-3.5-mini-instruct-Q4_K_M.gguf", "phi-3.5-mini", "phi-3.5-mini-instruct-q4_k_m.gguf"),
    ("phi-3.5-mini", "q3_k_m"): ("bartowski/Phi-3.5-mini-instruct-GGUF", "Phi-3.5-mini-instruct-Q3_K_M.gguf", "phi-3.5-mini", "phi-3.5-mini-instruct-q3_k_m.gguf"),
    ("phi-3.5-mini", "q2_k"):   ("bartowski/Phi-3.5-mini-instruct-GGUF", "Phi-3.5-mini-instruct-Q2_K.gguf",   "phi-3.5-mini", "phi-3.5-mini-instruct-q2_k.gguf"),
    # Mistral-7B
    ("mistral-7b", "q8_0"):   ("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q8_0.gguf",   "mistral-7b", "mistral-7b-instruct-v0.3-q8_0.gguf"),
    ("mistral-7b", "q6_k"):   ("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q6_K.gguf",   "mistral-7b", "mistral-7b-instruct-v0.3-q6_k.gguf"),
    ("mistral-7b", "q5_k_m"): ("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q5_K_M.gguf", "mistral-7b", "mistral-7b-instruct-v0.3-q5_k_m.gguf"),
    ("mistral-7b", "q4_k_m"): ("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q4_K_M.gguf", "mistral-7b", "mistral-7b-instruct-v0.3-q4_k_m.gguf"),
    ("mistral-7b", "q3_k_m"): ("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q3_K_M.gguf", "mistral-7b", "mistral-7b-instruct-v0.3-q3_k_m.gguf"),
    ("mistral-7b", "q2_k"):   ("bartowski/Mistral-7B-Instruct-v0.3-GGUF", "Mistral-7B-Instruct-v0.3-Q2_K.gguf",   "mistral-7b", "mistral-7b-instruct-v0.3-q2_k.gguf"),
}

# Quant size ordering for smallest-first processing
QUANT_ORDER = ["q2_k", "q3_k_m", "q3_k_l", "q4_k_m", "q5_k_m", "q6_k", "q8_0", "f16"]
# Model size ordering: 3B before 7B
MODEL_ORDER = ["qwen2.5-3b", "llama-3.2-3b", "phi-3.5-mini", "mistral-7b"]


def download_variant(model: str, quant: str) -> Path:
    """Download a single GGUF variant. Returns local path."""
    key = (model, quant)
    if key not in DOWNLOAD_MAP:
        raise ValueError(f"Unknown variant: {model}/{quant}")

    repo_id, hf_filename, local_dir, local_filename = DOWNLOAD_MAP[key]
    dest_dir = MODELS_DIR / local_dir
    dest_file = dest_dir / local_filename

    if dest_file.exists():
        print(f"  Already downloaded: {dest_file.name} ({dest_file.stat().st_size / 1e9:.1f} GB)")
        return dest_file

    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"  Downloading {repo_id}/{hf_filename}...")

    # Use a temp cache dir to avoid doubling disk usage
    cache_dir = MODELS_DIR / ".hf_cache"
    cache_dir.mkdir(exist_ok=True)

    path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=str(dest_dir),
        local_dir_use_symlinks=False,
        cache_dir=str(cache_dir),
    )

    # Rename if HF used original filename
    downloaded = Path(path)
    if downloaded.name != local_filename and downloaded.exists():
        downloaded.rename(dest_file)

    # Clean HF cache
    if cache_dir.exists():
        shutil.rmtree(cache_dir, ignore_errors=True)

    size_gb = dest_file.stat().st_size / 1e9
    print(f"  Downloaded: {dest_file.name} ({size_gb:.1f} GB)")
    return dest_file


def delete_variant(model: str, quant: str) -> None:
    """Delete GGUF file to free disk."""
    path = get_model_path(model, quant)
    if path.exists():
        size_gb = path.stat().st_size / 1e9
        path.unlink()
        print(f"  Deleted: {path.name} ({size_gb:.1f} GB freed)")


def get_ordered_variants() -> list[tuple[str, str]]:
    """All variants in smallest-first order."""
    variants = []
    for model in MODEL_ORDER:
        quants = MODEL_QUANTS.get(model, [])
        # Sort by QUANT_ORDER index
        sorted_quants = sorted(quants, key=lambda q: QUANT_ORDER.index(q) if q in QUANT_ORDER else 99)
        for quant in sorted_quants:
            if (model, quant) in DOWNLOAD_MAP:
                variants.append((model, quant))
    return variants


ordered = get_ordered_variants()
print(f"Total variants: {len(ordered)}")
for m, q in ordered:
    print(f"  {m}/{q}")

In [ ]:
# Cell 7: Main experiment loop
import csv
import gc
import time
import traceback
from run_experiments import (
    load_completed, append_result, run_single, truncate_to_tokens,
    RESULTS_CSV, CSV_COLUMNS,
)

MAX_CONSECUTIVE_ERRORS = 10

# Reload completed (may have resumed session)
completed = load_completed(RESULTS_CSV)
print(f"Already completed: {len(completed)} rows")

ordered_variants = get_ordered_variants()
total_pages = len(records)
total_experiments = len(ordered_variants) * total_pages
done_at_start = len(completed)

print(f"Total experiments: {total_experiments}")
print(f"Remaining: ~{total_experiments - done_at_start}")
print(f"="*60)

session_start = time.time()
session_done = 0

for vi, (model, quant) in enumerate(ordered_variants):
    # Check how many pages are already done for this variant
    variant_done = sum(1 for pid, m, q in completed if m == model and q == quant)
    if variant_done >= total_pages:
        print(f"\n[{vi+1}/{len(ordered_variants)}] {model}/{quant} — COMPLETE ({variant_done}/{total_pages})")
        continue

    print(f"\n[{vi+1}/{len(ordered_variants)}] {model}/{quant} — {variant_done}/{total_pages} done")

    # Download model
    try:
        download_variant(model, quant)
    except Exception as e:
        print(f"  DOWNLOAD FAILED: {e}")
        continue

    # Process pages
    consecutive_errors = 0
    variant_new = 0
    variant_start = time.time()

    for pi, page in enumerate(records):
        key = (page["id"], model, quant)
        if key in completed:
            continue

        try:
            result = run_single(page, model, quant, load_and_extract)
            if result:
                append_result(RESULTS_CSV, result)
                completed.add(key)
                consecutive_errors = 0
                variant_new += 1
                session_done += 1
            else:
                consecutive_errors += 1
        except RuntimeError as e:
            if "out of memory" in str(e).lower() or "oom" in str(e).lower():
                print(f"  GPU OOM at page {pi+1}! Skipping variant.")
                break
            consecutive_errors += 1
            print(f"  Error at page {pi+1}: {e}")
        except Exception as e:
            consecutive_errors += 1
            print(f"  Error at page {pi+1}: {e}")

        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            print(f"  {MAX_CONSECUTIVE_ERRORS} consecutive errors — skipping variant")
            break

        # Progress every 50 pages
        if variant_new > 0 and variant_new % 50 == 0:
            elapsed = time.time() - variant_start
            rate = variant_new / elapsed if elapsed > 0 else 0
            remaining = total_pages - variant_done - variant_new
            eta_min = remaining / rate / 60 if rate > 0 else 0
            print(f"  Progress: {variant_done + variant_new}/{total_pages} | {rate:.1f} pages/s | ETA: {eta_min:.0f}min")

    # Unload & delete
    unload_model()
    gc.collect()
    delete_variant(model, quant)

    elapsed = time.time() - variant_start
    print(f"  Variant done: +{variant_new} pages in {elapsed/60:.1f}min")

    # Session stats
    session_elapsed = (time.time() - session_start) / 3600
    total_done = done_at_start + session_done
    print(f"  Session: {session_done} new | {total_done}/{total_experiments} total ({100*total_done/total_experiments:.1f}%) | {session_elapsed:.1f}h elapsed")

print(f"\n{'='*60}")
print(f"Session complete: {session_done} new experiments in {(time.time()-session_start)/3600:.1f}h")
print(f"Total progress: {done_at_start + session_done}/{total_experiments}")

In [ ]:
# Cell 8: Results summary
import pandas as pd

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    print(f"Total rows: {len(df)}")
    print(f"Unique variants: {df.groupby(['model', 'quant']).ngroups}")
    print()

    # Per-variant summary
    summary = df.groupby(["model", "quant"]).agg(
        pages=pd.NamedAgg(column="page_id", aggfunc="count"),
        f1_mean=pd.NamedAgg(column="f1", aggfunc="mean"),
        f1_std=pd.NamedAgg(column="f1", aggfunc="std"),
        latency_mean=pd.NamedAgg(column="latency_s", aggfunc="mean"),
        json_valid_pct=pd.NamedAgg(column="json_valid", aggfunc="mean"),
        exact_match_pct=pd.NamedAgg(column="exact_match", aggfunc="mean"),
        model_size=pd.NamedAgg(column="model_size_gb", aggfunc="first"),
    ).round(4)

    summary["json_valid_pct"] = (summary["json_valid_pct"] * 100).round(1)
    summary["exact_match_pct"] = (summary["exact_match_pct"] * 100).round(1)

    # Sort by model order then quant order
    model_rank = {m: i for i, m in enumerate(MODEL_ORDER)}
    quant_rank = {q: i for i, q in enumerate(QUANT_ORDER)}
    summary["_model_rank"] = summary.index.get_level_values("model").map(model_rank)
    summary["_quant_rank"] = summary.index.get_level_values("quant").map(quant_rank)
    summary = summary.sort_values(["_model_rank", "_quant_rank"]).drop(columns=["_model_rank", "_quant_rank"])

    print(summary.to_string())

    # Completion status
    print(f"\n--- Completion ---")
    for model in MODEL_ORDER:
        for quant in MODEL_QUANTS.get(model, []):
            n = len(df[(df["model"] == model) & (df["quant"] == quant)])
            status = "DONE" if n >= 500 else f"{n}/500"
            print(f"  {model}/{quant}: {status}")
else:
    print("No results yet — run the main loop first.")

## Drive Persistence

Results are saved to **Google Drive** at `MyDrive/iitsrc/results/idea-c/experiments.csv`.

To continue after a session timeout:
1. Open this notebook again
2. Select T4 GPU runtime
3. **Run All** — it will skip completed experiments automatically

To download results locally:
```python
from google.colab import files
files.download(str(RESULTS_CSV))
```